# 
In the last notebook, you connected applications with Services and Ingress. In this notebook, you will learn two very popular ways to manage Kubernetes YAML at scale. Helm helps you **package** an application, and Kustomize helps you **customize** an existing set of manifests.

A beginner-friendly way to think about it is this:

- **Helm** is like a package manager plus templating system for Kubernetes
- **Kustomize** is like a layering tool for YAML patches


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain what a Helm chart is and why teams use Helm
- Add a Helm repository and install a chart into the cluster
- Inspect a Helm release and understand values and rendered manifests
- Create a simple custom Helm chart for our `api-gateway`
- Use different `values.yaml` files for dev and prod-style settings
- Render a chart locally with `helm template`
- Update a release with `helm upgrade`
- Explain what Kustomize is and when to use overlays
- Build a simple Kustomize base and a dev overlay
- Apply Kustomize resources with `kubectl apply -k`
- Compare Helm and Kustomize and know when each tool fits best


## 
       Before you start:

'Reload Window'.
       2. Make sure minikube is running.
       3. Make sure Helm is installed on your machine.
       4. Keep using the `k8s-lab` namespace from the previous notebooks.

       The next cell verifies your Kubernetes context and your Helm installation.


In [ ]:
!kubectl config current-context
!kubectl get namespace k8s-lab
!helm version


## 
Helm is the most common package manager for Kubernetes. If you have used `npm`, `pip`, or `apt`, you already understand the big idea: Helm lets you install reusable packages instead of copying and editing raw YAML by hand.

In Helm, those packages are called **charts**. A chart contains templates, default values, and metadata. When Helm installs a chart, it renders the templates into plain Kubernetes YAML and sends that YAML to the cluster.

Helm is especially helpful when:

- an app has many YAML files
- the same app needs different settings in dev and prod
- you want to reuse a deployment pattern many times


In [ ]:
!helm help | head -n 20


Add a Helm Repository        ## 

       A Helm repository is a place where charts are published. In this lab, we will add Bitnami's chart repository because it contains many well-known Kubernetes packages.

       After adding the repo, we will search for the Redis chart to prove that Helm can now discover it.


In [ ]:
!helm repo add bitnami https://charts.bitnami.com/bitnami --force-update
!helm repo update
!helm search repo bitnami/redis | head -n 5


## 
Let's install Redis from the Bitnami repo. Redis is a nice demo because it is a real application, but we can keep the setup simple by turning authentication and persistence off for the lab.

This command creates a Helm **release** named `my-redis` in the `k8s-lab` namespace.


In [ ]:
!helm install my-redis bitnami/redis --set auth.enabled=false --set master.persistence.enabled=false -n k8s-lab || helm upgrade --install my-redis bitnami/redis --set auth.enabled=false --set master.persistence.enabled=false -n k8s-lab
!kubectl get pods -n k8s-lab


## 
A Helm release stores more than just a name. Helm remembers the values you used and the manifests it rendered.

These commands answer three beginner-friendly questions:

- What releases are installed?
- What values did this release use?
- What YAML did Helm actually send to Kubernetes?


In [ ]:
!helm list -n k8s-lab
!helm get values my-redis -n k8s-lab
!helm get manifest my-redis -n k8s-lab | head -n 80


## 
Now you will create your own chart for the sample `api-gateway` service.

The original request mentioned `/tmp`, but in this lab we will use a project-local folder named `./helm-lab` so the generated files stay beside the notebook and are easier to inspect later.

`helm create` generates a starter chart with a common folder structure.


In [ ]:
!mkdir -p ./helm-lab
!rm -rf ./helm-lab/k8s-lab-chart
!helm create ./helm-lab/k8s-lab-chart
!find ./helm-lab/k8s-lab-chart -maxdepth 2 -type f | sort


## 
A generated Helm chart usually looks like this:

```text
k8s-lab-chart/
+-- Chart.yaml
+-- values.yaml
+-- charts/
+-- templates/
    +-- deployment.yaml
    +-- service.yaml
    +-- ingress.yaml
    +-- ...
```

- `Chart.yaml` describes the chart itself
- `values.yaml` stores default configuration values
- `templates/` contains Kubernetes YAML with placeholders


In [ ]:
!find ./helm-lab/k8s-lab-chart -maxdepth 2 -type f | sort


Customize the Chart for Our API Gateway        ## 

       The starter chart contains more files than we need for a beginner lab, so we will simplify it. First we remove the extra generated templates. Then we will replace them with a minimal Deployment and Service for `api-gateway`.

       This keeps the chart easy to read and easy to teach.


In [ ]:
!find ./helm-lab/k8s-lab-chart/templates -maxdepth 2 -type f | sort
!rm -f ./helm-lab/k8s-lab-chart/templates/*.yaml
!rm -rf ./helm-lab/k8s-lab-chart/templates/tests


In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/Chart.yaml
apiVersion: v2
name: k8s-lab-chart
description: Simple Helm chart for the Kubernetes lab api-gateway
type: application
version: 0.1.0
appVersion: "1.0.0"


In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/values.yaml
replicaCount: 1

image:
  repository: k8s-lab/api-gateway
  tag: latest
  pullPolicy: IfNotPresent

service:
  type: ClusterIP
  port: 8000
  targetPort: 8000

env:
  USER_SERVICE_URL: http://user-service:8001
  ORDER_SERVICE_URL: http://order-service:8002


In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/templates/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ .Release.Name }}
  namespace: {{ .Release.Namespace }}
  labels:
    app: {{ .Release.Name }}
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      app: {{ .Release.Name }}
  template:
    metadata:
      labels:
        app: {{ .Release.Name }}
    spec:
      containers:
        - name: api-gateway
          image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"
          imagePullPolicy: {{ .Values.image.pullPolicy }}
          ports:
            - containerPort: {{ .Values.service.targetPort }}
          env:
            - name: USER_SERVICE_URL
              value: {{ .Values.env.USER_SERVICE_URL | quote }}
            - name: ORDER_SERVICE_URL
              value: {{ .Values.env.ORDER_SERVICE_URL | quote }}


In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/templates/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: {{ .Release.Name }}
  namespace: {{ .Release.Namespace }}
  labels:
    app: {{ .Release.Name }}
spec:
  type: {{ .Values.service.type }}
  selector:
    app: {{ .Release.Name }}
  ports:
    - port: {{ .Values.service.port }}
      targetPort: {{ .Values.service.targetPort }}


## 
One of Helm's biggest strengths is that the chart stays the same while the values change. That lets you keep one chart but apply different settings for different environments.

In this lab, the dev version will use fewer replicas and a ClusterIP Service. The prod-style version will use more replicas and expose a NodePort.


In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/values-dev.yaml
replicaCount: 1

service:
  type: ClusterIP
  port: 8000
  targetPort: 8000


In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/values-prod.yaml
replicaCount: 3

service:
  type: NodePort
  port: 8000
  targetPort: 8000


## 
`helm template` is one of the best beginner tools because it shows you the final YAML before anything is applied to the cluster. That makes debugging much easier.

In other words, Helm templates first, then Kubernetes applies the rendered result.


In [ ]:
!helm template api-gateway-helm ./helm-lab/k8s-lab-chart -n k8s-lab -f ./helm-lab/k8s-lab-chart/values-dev.yaml | head -n 80


## 
`helm upgrade` takes a release that already exists and applies a new chart version or new values.

In the next cell, you will:

1. install or update a dev-style release named `api-gateway-helm`
2. inspect the Deployment
3. upgrade the same release with the prod-style values
4. inspect it again


In [ ]:
!helm upgrade --install api-gateway-helm ./helm-lab/k8s-lab-chart -n k8s-lab -f ./helm-lab/k8s-lab-chart/values-dev.yaml
!kubectl get deploy,svc -n k8s-lab | grep api-gateway-helm
!helm upgrade api-gateway-helm ./helm-lab/k8s-lab-chart -n k8s-lab -f ./helm-lab/k8s-lab-chart/values-prod.yaml
!kubectl get deploy api-gateway-helm -n k8s-lab -o wide


## 
Kustomize solves a different problem from Helm. Instead of writing templates, you start with plain YAML and then apply small changes called **patches**.

This is very handy when you already have manifests and want lightweight environment overlays.

```text
Base YAML
   |
   v
+-----------+
| Overlay   |  <- small patch for dev, prod, staging
+-----------+
   |
   v
Final YAML
```

Helm is stronger when you want reusable packaged applications. Kustomize is great when you want to layer changes onto existing manifests.


In [ ]:
!kubectl kustomize --help | head -n 20


## 
We will build a tiny Kustomize example from scratch.

- The **base** will define a simple `api-gateway-kustomize` Deployment and Service
- The **dev overlay** will patch the Deployment to change the replica count and add a `LAB_ENV` variable

This is a very common Kustomize pattern.


In [ ]:
!mkdir -p ./kustomize-lab/base ./kustomize-lab/overlays/dev


In [ ]:
%%writefile ./kustomize-lab/base/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: api-gateway-kustomize
spec:
  replicas: 1
  selector:
    matchLabels:
      app: api-gateway-kustomize
  template:
    metadata:
      labels:
        app: api-gateway-kustomize
    spec:
      containers:
        - name: api-gateway
          image: k8s-lab/api-gateway:latest
          ports:
            - containerPort: 8000
          env:
            - name: USER_SERVICE_URL
              value: http://user-service:8001
            - name: ORDER_SERVICE_URL
              value: http://order-service:8002


In [ ]:
%%writefile ./kustomize-lab/base/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: api-gateway-kustomize
spec:
  type: ClusterIP
  selector:
    app: api-gateway-kustomize
  ports:
    - port: 8000
      targetPort: 8000


In [ ]:
%%writefile ./kustomize-lab/base/kustomization.yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization
resources:
  - deployment.yaml
  - service.yaml


In [ ]:
%%writefile ./kustomize-lab/overlays/dev/patch.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: api-gateway-kustomize
spec:
  replicas: 2
  template:
    spec:
      containers:
        - name: api-gateway
          env:
            - name: USER_SERVICE_URL
              value: http://user-service:8001
            - name: ORDER_SERVICE_URL
              value: http://order-service:8002
            - name: LAB_ENV
              value: dev


In [ ]:
%%writefile ./kustomize-lab/overlays/dev/kustomization.yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization
namespace: k8s-lab
resources:
  - ../../base
patches:
  - path: patch.yaml
    target:
      kind: Deployment
      name: api-gateway-kustomize


Apply with `kubectl apply -k`        ## 

       `kubectl apply -k` tells Kubernetes to render the Kustomize overlay first and then apply the final YAML.

       This is the simplest way to use Kustomize because you do not need a separate binary when `kubectl` already supports it.


In [ ]:
!kubectl apply -k ./kustomize-lab/overlays/dev
!kubectl get deploy,svc -n k8s-lab | grep api-gateway-kustomize


Compare Helm vs Kustomize        ## 

       Here is a simple rule of thumb:

       - Use **Helm** when you want a reusable packaged application with values and release history
       - Use **Kustomize** when you already have YAML and want to patch it with overlays

       The next cell renders both approaches so you can compare the generated output side by side.


In [ ]:
!echo "Helm render:" && helm template api-gateway-helm ./helm-lab/k8s-lab-chart -n k8s-lab -f ./helm-lab/k8s-lab-chart/values-dev.yaml | head -n 25
!echo ""
!echo "Kustomize render:" && kubectl kustomize ./kustomize-lab/overlays/dev | head -n 25


## 
The next cell removes the demo resources created by this notebook. We keep the local chart and overlay files on disk so you can continue reading them after the lab.


In [ ]:
!helm uninstall my-redis -n k8s-lab || true
!helm uninstall api-gateway-helm -n k8s-lab || true
!kubectl delete -k ./kustomize-lab/overlays/dev --ignore-not-found


## 
In this notebook, you learned that:

- **Helm** packages Kubernetes applications into reusable charts
- A Helm **release** is an installed chart with its own values and history
- `helm repo add`, `helm install`, `helm get`, `helm template`, and `helm upgrade` are core daily commands
- A custom chart can stay small and beginner-friendly by keeping only the templates you need
- Different values files let one chart behave differently in dev and prod-style environments
- **Kustomize** works by starting with base YAML and layering patches on top
- `kubectl apply -k` is the easiest way to use Kustomize in everyday workflows
- Helm and Kustomize both solve configuration problems, but they do it in different ways

If these ideas feel comfortable now, you are ready for the next stage of the lab series where Kubernetes becomes easier to operate at scale.
